# 第63课 · 用「看图」的方式听懂关键词——CNN 遇见 Mel 频谱，从零定义 KeywordCNN

**目标**：设计 CNN 分类器，输入 mel 特征图 `(B, 1, n_mels, T)`，输出 `n_classes` 类的 logit。

🔗 **Aurora 连接**：输入特征由 `aurora.audio.mel` 的 `mel_spectrogram()` 产出；本节是 Aurora 第一个完整的 ML 应用，将 DSP 特征与 PyTorch 模型直接打通。

← **上一课**　[L62 · Dataset 与 DataLoader](L62_kws_dataset.ipynb)

> 上节课学习了 **Dataset 与 DataLoader**：自定义 `__getitem__`，把音频批量加载为 `(n_mels=40, T=32)` 的 mel 特征图。  
> 本课将探讨 **音频分类模型**。

## 学习目标

完成本课后，你将能够：

1. **解释** 为何 `kernel_size=(n_mels, k)` 能在单次卷积中压缩频率维度，将 mel 特征图转换为 1D 时间序列。
2. **实现** `KeywordCNN.forward` 方法，完成 conv1 → ReLU → conv2 → ReLU → GAP → fc 的完整前向传播。
3. **验证** Global Average Pooling 使模型对任意输入长度 `T` 都能正确运行。
4. **分析** 感受野与 kernel 宽度的定量关系：两层叠加后 `RF = k1 + k2 - 1`。


## 本课剧情：为什么用"看图"的方法来"听话"？

有没有想过，Shazam 识曲、Siri 听懂"Hey Siri"，背后用的不是某种神奇的"声音理解"，而是把声音变成图片，再用图像识别技术分类？

**Mel 频谱图就是声音的"指纹地图"**：

```
横轴 = 时间帧 (T=32帧, hop=512≈32ms，沿用 L62 数据流水线的设定)
纵轴 = Mel 频段 (n_mels=40)
像素 = 能量大小 (dB)
```

CNN 天然擅长在这类二维网格上抓局部模式：

| CNN 层 | 感受野 | 捕捉什么 |
|---|---|---|
| 第1层 `(40×5)` | 全频段×5帧 | 单个音素的频率轮廓 |
| 第2层 `(1×5)` | 时间上5帧 | 音素序列的时间模式 |
| GAP | 整段音频 | 全局特征向量 → 分类 |

**关键设计**：第一个卷积核高度 = `n_mels=40`，一次卷积**把整个频率维压成 1**，输出变成 `(B, 32, 1, T')`，后续只在时间轴上做 1D 特征提取。

```
输入 (B, 1, 40, 32)
   ↓ Conv2d(1→32, kernel=(40,5)) → (B, 32, 1, 28)
   ↓ ReLU
   ↓ Conv2d(32→64, kernel=(1,5)) → (B, 64, 1, 24)
   ↓ ReLU
   ↓ Global Average Pool → (B, 64)
   ↓ Linear(64, n_classes) → (B, 10)
```

本节任务：实现 `KeywordCNN`，让 `m(randn(8,1,40,32)).shape == (8,10)`。

In [ ]:
import torch
import torch.nn as nn

## 1. 输入 Shape：`(B, 1, n_mels, T)`

PyTorch 的 `nn.Conv2d` 期望输入格式为 `(batch, channels, height, width)`。对 mel 特征图：

- `B`：batch size
- `1`：单通道（灰度图，对应单帧能量）
- `n_mels`：Mel 频段数，作为「高度」
- `T`：时间帧数，作为「宽度」

典型值：`n_mels=40`，1秒音频 @ 16kHz。时间帧数 T 取决于 hop_length：

- hop=200，center padding：`T = floor(16000/200) + 1 = 81`（本课代码示例使用此设定）
- hop=160，center padding：`T = floor(16000/160) + 1 = 101`（另一常见配置）

注意 `T` 在推理时可变，模型必须对任意 `T` 都能运行，这是设计 global pooling 的动机。

In [ ]:
# 模拟一批 mel 特征图
B, n_mels, T = 8, 40, 81
x = torch.randn(B, 1, n_mels, T)
print(f'输入 shape: {x.shape}')  # torch.Size([8, 1, 40, 81])

## 2. 1D 时间卷积：`Conv2d(1, 32, kernel_size=(n_mels, 5))`

卷积核（convolution kernel）高度等于 `n_mels`（覆盖全部 Mel 维度），宽度为 5（滑动时间窗）。一次卷积后，频率维从 `n_mels` 压缩到 `1`，时间维保持（配合 `padding=(0,2)`）：

```
输入:  (B, 1,  n_mels, T)
conv1: (B, 32, 1,      T)   ← 频率维完全压缩，时间维不变
```

这等价于在时间轴上做 1D 卷积，但同时对所有 Mel 频段做加权求和。感受野（receptive field，RF）宽度 = kernel_width + (layers-1)×(kernel_width-1)，单层时就是 5 帧。

In [ ]:
# 验证 conv1 输出 shape
n_mels = 40
conv1 = nn.Conv2d(1, 32, kernel_size=(n_mels, 5), padding=(0, 2))
x = torch.randn(8, 1, n_mels, 81)
y = conv1(x)
print(f'conv1 输出 shape: {y.shape}')  # torch.Size([8, 32, 1, 81])
# 频率维 → 1，时间维 81 不变（padding=(0,2) 补偿 kernel宽5 的边缘）

## 3. Global Average Pooling：把时间维平均掉

分类任务需要固定长度的特征向量，但 `T` 在推理时可变。Global Average Pooling 对时间轴取平均：

```
(B, 64, 1, T) → mean(dim=-1) → (B, 64, 1) → squeeze() → (B, 64)
```

这比 Flatten + 全连接更鲁棒：参数量与 `T` 无关，且对时间平移有一定不变性。之后接 `nn.Linear(64, n_classes)` 输出每类的 logit，再配合 `CrossEntropyLoss` 训练。

In [ ]:
# 演示 Global Average Pooling
feat = torch.randn(8, 64, 1, 81)   # conv 输出
pooled = feat.mean(dim=-1).squeeze(dim=-1)  # (8, 64)
print(f'池化后 shape: {pooled.shape}')  # torch.Size([8, 64])

# 不同 T 得到相同输出 shape
feat2 = torch.randn(8, 64, 1, 150)
pooled2 = feat2.mean(dim=-1).squeeze(dim=-1)
print(f'T=150 池化后 shape: {pooled2.shape}')  # torch.Size([8, 64])

## 4. ✏️ 实现 `class KeywordCNN(nn.Module)`

**架构（4层）**：

| 层 | 类型 | 参数 | 输出 shape |
|---|---|---|---|
| conv1 | Conv2d | in=1, out=32, kernel=(40,5) | (B,32,1,T-4) |
| conv2 | Conv2d | in=32, out=64, kernel=(1,5) | (B,64,1,T-8) |
| GAP | AdaptiveAvgPool2d | output=(1,1) | (B,64,1,1) |
| fc | Linear | 64→n_classes | (B,n_classes) |

**实现要点**：

```python
def forward(self, x):
    x = F.relu(self.conv1(x))   # (B,1,40,81) → (B,32,1,77)
    x = F.relu(self.conv2(x))   # → (B,64,1,73)
    x = self.pool(x)             # → (B,64,1,1)
    x = x.view(x.size(0), -1)   # → (B,64)
    return self.fc(x)            # → (B,n_classes)
```

**验收标准**：
- `KeywordCNN(40,10)(torch.randn(8,1,40,81)).shape == (8,10)`
- `sum(p.numel() for p in m.parameters())` 合理（约 5k-15k 参数）

In [ ]:
class KeywordCNN(nn.Module):
    def __init__(self, n_mels=40, n_classes=10):
        super().__init__()
        # ✏️ TODO: 定义 conv1, conv2, pool, fc
        raise NotImplementedError("TODO: 定义 conv1, conv2, pool, fc 各层")

    def forward(self, x):
        # ✏️ TODO: 实现前向传播，返回 (B, n_classes) logit
        raise NotImplementedError("TODO: 实现前向传播，返回 (B, n_classes) logit")

In [ ]:
# 检查：输出 shape 必须为 (8, 10)
try:
    m = KeywordCNN(40, 10)
    out = m(torch.randn(8, 1, 40, 81))
    assert out.shape == (8, 10), f'期望 (8,10)，实际 {out.shape}'
    print('✅ KeywordCNN 输出 shape 正确：', out.shape)

    # 额外检查：不同 T 也能运行（Global Pooling 的核心优势）
    out2 = m(torch.randn(4, 1, 40, 150))
    assert out2.shape == (4, 10)
    print('✅ 可变时间长度 T=150 也通过：', out2.shape)

    # 额外检查：n_classes=35（Speech Commands 全集大小，防止 fc 层硬编码 10）
    out3 = KeywordCNN(40, 35)(torch.randn(2, 1, 40, 81))
    assert out3.shape == (2, 35), f'期望 (2,35)，实际 {out3.shape}'
    print('✅ n_classes=35 也通过：', out3.shape)
except NotImplementedError as e:
    print(f'⚠️  KeywordCNN 尚未实现，请完成 TODO 后重新运行本格。提示: {e}')

## 5. 参数实验：时间卷积核大小对感受野的影响

**实验变量**：`conv1` 的时间维 kernel 宽度（3 vs 7）

单层感受野 = kernel_width；两层叠加后感受野 = k1 + k2 - 1。

```
kernel=3: conv1感受野=3帧, conv2感受野=3帧 → 总计 3+3-1=5帧
kernel=7: conv1感受野=7帧, conv2感受野=3帧 → 总计 7+3-1=9帧
```

**预期现象**：kernel=7 的模型对较长音素模式（如元音）响应更强，kernel=3 对短促辅音更敏感。实际验证需用真实语音数据，这里只比较参数量和激活范数。

In [ ]:
def make_cnn_with_k(k):
    """用不同时间 kernel 宽度构造 KeywordCNN（修改 conv1 的 kernel）"""
    n_mels = 40
    net = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=(n_mels, k), padding=(0, k//2)),
        nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d((1, 2)),
    )
    return net

x = torch.randn(8, 1, 40, 81)
for k in [3, 5, 7]:
    net = make_cnn_with_k(k)
    params = sum(p.numel() for p in net.parameters())
    out = net(x)
    # 用激活范数粗估响应强度
    norm = out.norm().item()
    receptive_field = k + 3 - 1  # conv1 + conv2 的感受野
    print(f'kernel={k}: 参数量={params:,}, 时间感受野={receptive_field}帧, 激活范数={norm:.2f}')

## 本课收束

`KeywordCNN` 通过 `conv1`（全频率压缩）+ `conv2`（特征精炼）+ Global Average Pooling，将任意长度的 mel 特征图映射为 `(B, n_classes)` logit 向量。输入特征直接来自 `aurora.audio.mel.mel_spectrogram()`，两者无缝对接，构成 Aurora 第一个端到端的 ML pipeline。下一节（L64）将在 Google Speech Commands 子集上训练该模型，观察 loss 曲线与分类准确率。

## ✏️ 白板挑战：CNN 架构手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：`Conv2d(1, 32, kernel=(40,5))` 作用于 `(B,1,40,81)` 输入，输出 shape 是什么？  
（高: 40-40+1=1, 宽: 81-5+1=77 → (B,32,1,77)）

**问 2**：为什么 conv1 的 kernel 高度设为 `n_mels=40`（而不是 3 或 5）？  
（一次压缩整个频率维，把频率信息聚合进 32 个通道，后续只需在时间轴做1D操作）

**问 3**：`AdaptiveAvgPool2d(output_size=(1,1))` 对任意 T 的输入输出什么 shape？  
（对任意 (B,C,1,T) → (B,C,1,1)；解决了变长序列问题，不依赖具体 T 值）

**问 4**：`KeywordCNN(40,10)` 总参数数是多少？逐层计算。  
（conv1: (40×5×1+1)×32=6432; conv2: (1×5×32+1)×64=10304; fc: 64×10+10=650 → 约17386）

**问 5**：如果把 `Global Average Pool` 换成 `Flatten()`（不平均，直接展开），会有什么问题？  
（输出长度依赖T，不同长度音频输出维度不同，fc层权重无法复用；同时参数量爆炸）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import torch, torch.nn as nn

# 问1：Conv2d输出shape
conv1_q = nn.Conv2d(1, 32, kernel_size=(40, 5))
x_q = torch.randn(8, 1, 40, 81)
out1_q = conv1_q(x_q)
assert out1_q.shape == (8, 32, 1, 77), f"shape={out1_q.shape}"
print(f"Q1 ✅  Conv2d(1,32,(40,5)) + (B,1,40,81) → {tuple(out1_q.shape)}")

# 问2：频率维压缩（概念验证）
assert out1_q.shape[2] == 1, "高度维应=1（全频率压缩）"
print(f"Q2 ✅  kernel高=n_mels=40，卷积后高度维={out1_q.shape[2]}（完全压缩频率轴）")

# 问3：AdaptiveAvgPool对变长T
pool_q = nn.AdaptiveAvgPool2d((1, 1))
for T_test in [77, 50, 32]:
    feat_q = torch.randn(4, 64, 1, T_test)
    pooled_q = pool_q(feat_q)
    assert pooled_q.shape == (4, 64, 1, 1), f"T={T_test}: shape={pooled_q.shape}"
print(f"Q3 ✅  AdaptiveAvgPool2d((1,1)): T=77/50/32均→(B,64,1,1)，与T无关")

# 问4：KeywordCNN参数总量
try:
    m_q = KeywordCNN(40, 10)
    total_q = sum(p.numel() for p in m_q.parameters())
    print(f"Q4 ✅  KeywordCNN(40,10) 参数总量={total_q}")
    # conv1: (40*5*1+1)*32=6432, conv2: (1*5*32+1)*64=10304, fc: 64*10+10=650
    assert total_q == 17386, f"期望17386，实际{total_q}"
except NotImplementedError:
    # 手算验证
    c1 = (40*5*1+1)*32
    c2 = (1*5*32+1)*64
    fc = 64*10+10
    print(f"Q4 ✅  逐层: conv1={c1}, conv2={c2}, fc={fc}, 共{c1+c2+fc}（待实现后验证）")

# 问5：GAP vs Flatten（概念验证）
feat_77 = torch.randn(2, 64, 1, 77)
feat_50 = torch.randn(2, 64, 1, 50)
gap = nn.AdaptiveAvgPool2d((1, 1))
assert gap(feat_77).view(2,-1).shape == gap(feat_50).view(2,-1).shape == (2, 64)
print(f"Q5 ✅  GAP输出始终(B,64)，Flatten则T=77→{77*64}，T=50→{50*64}，长度不一致")
print("\n🎉 CNN架构白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l63_review = {
    "mel_as_image_understood":  None,  # 理解Mel频谱=时频图，CNN适合处理的原因？True/False
    "freq_compress_design":     None,  # 理解conv1 kernel高=n_mels，一次压缩整个频率维？True/False
    "gap_vs_flatten":           None,  # 理解Global Average Pool解决变长T的原因？True/False
    "keyword_cnn_impl":         None,  # KeywordCNN实现正确，shape/参数验证通过？True/False
    "whiteboard_passed":        None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l63_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l63_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L63 全部通关！进入 L64：训练评估闭环')

---

→ **下一课**　[L64 · 训练评估闭环](L64_kws_train_eval.ipynb)

> 下节课将学习 **训练评估闭环**：train loop + val loop，准确率 / 混淆矩阵 / 过拟合诊断。